# Module 01 — Spark API & Query Plans

Goals: PySpark DataFrame API vs `spark.sql`, one engine underneath; read plans from logical → physical.

Open the **Application UI** (host port mapped to container `4040`) before running actions.

In [ ]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
from shared.demo_data import user_post_activity

spark = begin_module('module-01-spark-plans')
ui_links()

## TLC data

In [ ]:
trips = spark.read.parquet(*config.yellow_taxi_paths())
zones = spark.read.option('header', True).csv(config.zones_path())
print('trips columns:', len(trips.columns))

## PySpark vs Spark SQL — same query

Spark parses both into a **logical plan**, runs **analyzer** (resolve columns/types), **optimizer** (rules), then builds a **physical plan** for execution.

In [ ]:
from pyspark.sql import functions as F

df_api = (
    trips.filter(F.col('VendorID').isin(1, 2))
    .groupBy('VendorID')
    .agg(F.count('*').alias('trips'), F.avg('total_amount').alias('avg_fare'))
    .orderBy(F.desc('trips'))
)
df_api.show(5)

In [ ]:
df_sql = spark.sql('''
    SELECT VendorID, COUNT(*) AS trips, AVG(total_amount) AS avg_fare
    FROM parquet.`{}`
    WHERE VendorID IN (1, 2)
    GROUP BY VendorID
    ORDER BY trips DESC
'''.format(config.yellow_taxi_paths()[0]))
df_sql.show(5)

In [ ]:
print('--- DataFrame API plan ---')
df_api.explain('formatted')
print('--- Spark SQL plan ---')
df_sql.explain('formatted')

## Extended `explain(true)` — plan phases

In [ ]:
df_api.explain(mode='extended')

### Vocabulary

| Phase | Role |
|-------|------|
| **Parsed / Logical** | What you asked for |
| **Analyzed** | Catalog resolved, types fixed |
| **Optimized** | Filter pushdown, column pruning (more in module 02) |
| **Physical** | Joins, shuffle, scans chosen for executors |

## Example 2 — projection + filter (DataFrame & SQL)

In [ ]:
narrow_api = trips.select('VendorID', 'total_amount', 'PULocationID').filter('total_amount > 50')
narrow_sql = spark.sql(f'''
    SELECT VendorID, total_amount, PULocationID
    FROM parquet.`{config.yellow_taxi_paths()[0]}`
    WHERE total_amount > 50
''')
explain_plan(narrow_api)
narrow_sql.explain('formatted')

## Example 3 — join then aggregate

In [ ]:
from pyspark.sql import functions as F
z = zones.withColumnRenamed('LocationID', 'PULocationID')
joined_api = trips.join(z, 'PULocationID', 'left')
by_borough = joined_api.groupBy('Borough').agg(F.count('*').alias('n'))
by_borough.show(5)
explain_plan(by_borough)

## Spark UI: jobs, stages, tasks

Run `by_borough.show()` — note the **job** in the UI. Transformations alone only extend the plan.

In [ ]:
spark.conf.set('spark.sql.shuffle.partitions', '8')
by_borough.explain('formatted')
spark.conf.set('spark.sql.shuffle.partitions', '200')
by_borough.explain('formatted')

## Takeaways

1. DataFrame API and Spark SQL converge on the same planner.
2. Use `explain('formatted')` in class; `extended` shows analyzer/optimizer output.
3. Physical plan + UI connects code to cluster work (modules 02–05 go deeper).